# Backtracking

## 1. Backtracking Fundamentals

**Recursion** solves a problem by breaking it into smaller subproblems of the same structure. Each recursive call handles one subproblem, while the call stack tracks the sequence of calls, each waiting for the next to return.

- **Backtracking:** extends recursion with a specific strategy. Build a candidate solution one decision at a time, and abandon the current path as soon as a constraint is violated.
- **Pruning:** the act of abandoning a path and skipping its entire subtree. The abandoned subtree is never explored.

Every backtracking problem has three elements:

| Element | Role |
|---------|------|
| **Choice** | What decision is made at each recursive step |
| **Constraint** | What condition, if violated, causes the current path to be abandoned |
| **Goal** | What defines a complete, valid solution |

### State Space Tree

A **state space tree** represents the full space of possible decisions:

- **Root:** the empty starting state, before any decisions
- **Edges:** each edge corresponds to one specific choice
- **Internal nodes:** partial candidates, with some decisions made and others remaining
- **Leaves:** terminal states, either valid solutions (goal met) or dead ends (constraint violated)

```mermaid
graph TD
    R["root"] -->|"choice a"| A["a"]
    R -->|"choice b"| B["b"]
    R -->|"choice c"| C["c (pruned)"]
    A -->|"choice d"| D["d"]
    A -->|"choice e"| E["e (pruned)"]
    B -->|"choice f"| F["f"]
    D --> G["solution"]
    F --> H["solution"]
```

- **Pruned nodes** (`c`, `e`): constraint violated at that node. The entire subtree below is skipped.
- **Solution nodes** (`d -> solution`, `f -> solution`): goal reached. The path from root to this node is recorded.

### Complexity

- **Branching factor** $b$: the number of choices available at each node. In the tree above, the root has 3 children (a, b, c), so $b = 3$ at that level.
- **Depth** $d$: maximum number of decisions to reach a leaf
- **Worst-case time**: $O(b^d)$, visiting every node in the state space tree
- **Space**: $O(d)$ for the recursion stack (one path from root to leaf at a time)

Pruning eliminates some choices at each node, so fewer than $b$ branches are explored at most nodes. Actual runtime is often much less than $O(b^d)$.

### Backtracking vs DFS vs DP

| Approach | Explores | Pruning | Caching |
|----------|----------|---------|---------|
| DFS | Entire tree | Only at leaves | No |
| Backtracking | Pruned tree | At any node where constraints fail | No |
| Top-down DP | Pruned tree | At any node | Yes (memoization) |

- DFS is a special case of backtracking where pruning only occurs at leaf nodes.
- Top-down DP adds **memoization** to backtracking:
  - Memoization stores the result of each state (e.g., in a dictionary keyed by the state) so that if the same state is reached again through a different sequence of decisions, the stored result is returned instead of recomputing it. This is useful when many different decision paths lead to the same subproblem.
  - In problems like N-Queens, each board configuration is essentially unique because the exact placement of queens differs at every node, so almost no state is ever revisited.
  - Each call inserts a state into the dictionary and checks the dictionary before recursing. Since no state repeats, every check misses and every insert is wasted.
  - The dictionary accumulates millions of entries that are never looked up again, adding memory for storage and time for hashing and lookup on every call, with no benefit. Backtracking without memoization avoids this entirely.

### Properties

| Property | Detail |
|---|---|
| **Key idea** | Build candidates incrementally, abandon as soon as constraints fail |
| **Three elements** | Choice, Constraint, Goal |
| **State space** | Tree of depth $d$ with branching factor $b$ |
| **Time** | $O(b^d)$ worst case, reduced by pruning |
| **Space** | $O(d)$ for recursion stack |

## 2. Template

The standard backtracking template follows a **make-choice / recurse / undo-choice** pattern:

1. Check if the current state satisfies the **goal**. If so, record it and return.
2. Iterate over available **choices**.
3. For each choice, check if it satisfies **constraints**. If not, skip (prune).
4. **Make** the choice (modify state).
5. **Recurse** to the next decision level.
6. **Undo** the choice (restore state to what it was before step 4).

The undo step is essential. The `path` list is shared across all branches of the recursion. If choice A appends a value and does not remove it after recursing, choice B sees A's value still in `path` and builds on top of it, producing paths that contain both A and B when they should be separate branches. Undoing (e.g., `path.pop()`) removes A's value so B starts from the same `path` that A started from.

In [1]:
def backtrack(path, choices, res):
    if len(path) == 2:
        res.append(path[:])
        return
    for c in choices:
        path.append(c)
        backtrack(path, choices, res)
        path.pop()


res = []
backtrack([], [1, 2], res)
print(res)

[[1, 1], [1, 2], [2, 1], [2, 2]]


#### Steps

`backtrack([], [1, 2], res)` — generate all paths of length 2 from choices [1, 2].

**State space tree:**

```mermaid
graph TD
    A["path=[]"] -->|"append 1"| B["path=[1]"]
    A -->|"append 2"| C["path=[2]"]
    B -->|"append 1"| D["[1,1] *"]
    B -->|"append 2"| E["[1,2] *"]
    C -->|"append 1"| F["[2,1] *"]
    C -->|"append 2"| G["[2,2] *"]
```

`*` = len == 2, path recorded. 4 leaves in a tree with branching factor 2 and depth 2, matching $b^d = 2^2 = 4$.

**Undo trace:**

| Step | Action | path |
|------|--------|------|
| 1 | append 1 | [1] |
| 2 | append 1, record [1,1] | [1, 1] |
| 3 | undo (pop) | [1] |
| 4 | append 2, record [1,2] | [1, 2] |
| 5 | undo (pop) | [1] |
| 6 | undo (pop) | [] |
| 7 | append 2 | [2] |
| 8 | append 1, record [2,1] | [2, 1] |
| 9 | undo (pop) | [2] |
| 10 | append 2, record [2,2] | [2, 2] |
| 11 | undo (pop) | [2] |
| 12 | undo (pop) | [] |

Result: `[[1, 1], [1, 2], [2, 1], [2, 2]]`

## 3. Subsets

Given a list of $n$ distinct integers, return all $2^n$ subsets (the powerset).

- **Choice:** at each index $i$, include or exclude `nums[i]` -- a binary decision
- **Constraint:** none (every path is valid)
- **Goal:** index reaches $n$ (all elements considered)

The state space tree has branching factor 2 and depth $n$, producing $2^n$ leaves.

- Time: $O(n \cdot 2^n)$ -- $2^n$ subsets, each copied in $O(n)$
- Space: $O(n)$ recursion depth

In [2]:
def subsets(nums):
    res = []

    def backtrack(i, path):
        if i == len(nums):
            res.append(path[:])
            return
        path.append(nums[i])
        backtrack(i + 1, path)
        path.pop()
        backtrack(i + 1, path)

    backtrack(0, [])
    return res


print(subsets([1, 2, 3]))

[[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


#### Steps

`subsets([1, 2, 3])` — generate all $2^3 = 8$ subsets.

**State space tree** (left = include, right = exclude):

```mermaid
graph TD
    A["i=0, path=[]"] -->|"incl 1"| B["i=1, path=[1]"]
    A -->|"excl 1"| C["i=1, path=[]"]
    B -->|"incl 2"| D["i=2, path=[1,2]"]
    B -->|"excl 2"| E["i=2, path=[1]"]
    C -->|"incl 2"| F["i=2, path=[2]"]
    C -->|"excl 2"| G["i=2, path=[]"]
    D -->|"incl 3"| H["[1,2,3] *"]
    D -->|"excl 3"| I["[1,2] *"]
    E -->|"incl 3"| J["[1,3] *"]
    E -->|"excl 3"| K["[1] *"]
    F -->|"incl 3"| L["[2,3] *"]
    F -->|"excl 3"| M["[2] *"]
    G -->|"incl 3"| N["[3] *"]
    G -->|"excl 3"| O["[] *"]
```

`*` = i == 3, path recorded. 8 leaves from a binary tree of depth 3.

**Undo trace** (first two leaves):

| Step | Action | path |
|------|--------|------|
| 1 | incl 1 (append) | [1] |
| 2 | incl 2 (append) | [1, 2] |
| 3 | incl 3 (append), record [1,2,3] | [1, 2, 3] |
| 4 | undo 3 (pop) | [1, 2] |
| 5 | excl 3, record [1,2] | [1, 2] |
| 6 | undo 2 (pop) | [1] |
| 7 | excl 2, incl 3 (append), record [1,3] | [1, 3] |
| ... | continues for remaining 5 subsets | ... |

Result: `[[1,2,3], [1,2], [1,3], [1], [2,3], [2], [3], []]`

## 4. Permutations

Given $n$ distinct integers, return all $n!$ permutations.

- **Choice:** at each position, pick one unused element
- **Constraint:** each element used at most once (tracked by a `used` boolean array)
- **Goal:** path length reaches $n$

Branching factor decreases each level: $n, n-1, \ldots, 1$, yielding $n!$ leaves.

- Time: $O(n \cdot n!)$ -- $n!$ permutations, each copied in $O(n)$
- Space: $O(n)$ for path + used array + recursion stack

In [3]:
def permutations(nums):
    res = []
    used = [False] * len(nums)

    def backtrack(path):
        if len(path) == len(nums):
            res.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack(path)
            path.pop()
            used[i] = False

    backtrack([])
    return res


print(permutations([1, 2, 3]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


#### Steps

`permutations([1, 2, 3])` — generate all $3! = 6$ permutations.

**State space tree:**

```mermaid
graph TD
    A["path=[]"] -->|"append 1"| B["path=[1]"]
    A -->|"append 2"| C["path=[2]"]
    A -->|"append 3"| D["path=[3]"]
    B -->|"append 2"| E["path=[1,2]"]
    B -->|"append 3"| F["path=[1,3]"]
    C -->|"append 1"| G["path=[2,1]"]
    C -->|"append 3"| H["path=[2,3]"]
    D -->|"append 1"| I["path=[3,1]"]
    D -->|"append 2"| J["path=[3,2]"]
    E -->|"append 3"| K["[1,2,3] *"]
    F -->|"append 2"| L["[1,3,2] *"]
    G -->|"append 3"| M["[2,1,3] *"]
    H -->|"append 1"| N["[2,3,1] *"]
    I -->|"append 2"| O["[3,1,2] *"]
    J -->|"append 1"| P["[3,2,1] *"]
```

`*` = goal reached, path recorded.

**Undo trace** (first branch only):

| Step | Action | path | used |
|------|--------|------|------|
| 1 | append 1 | [1] | [T, F, F] |
| 2 | append 2 | [1, 2] | [T, T, F] |
| 3 | append 3, record [1,2,3] | [1, 2, 3] | [T, T, T] |
| 4 | undo 3 (pop) | [1, 2] | [T, T, F] |
| 5 | undo 2 (pop) | [1] | [T, F, F] |
| 6 | append 3 | [1, 3] | [T, F, T] |
| 7 | append 2, record [1,3,2] | [1, 3, 2] | [T, T, T] |
| 8 | undo 2 (pop) | [1, 3] | [T, F, T] |
| 9 | undo 3 (pop) | [1] | [T, F, F] |
| 10 | undo 1 (pop) | [] | [F, F, F] |

After step 10, the recursion continues with append 2 at the root, producing [2,1,3] and [2,3,1], then append 3, producing [3,1,2] and [3,2,1].

Result: `[[1,2,3], [1,3,2], [2,1,3], [2,3,1], [3,1,2], [3,2,1]]`

## 5. Combinations

Given integers $n$ and $k$, return all combinations of $k$ numbers from $\{1, \ldots, n\}$.

- **Choice:** at each level, pick the next number from `start` to $n$
- **Constraint:** `start` increments after each pick, preventing duplicate combinations (e.g., [2,1] is never generated because once 1 is skipped, only numbers > 1 are considered)
- **Goal:** path length reaches $k$
- **Pruning:** if fewer elements remain than slots to fill (`n - i + 1 < k - len(path)`), skip

- Time: $O(C(n,k) \cdot k)$ -- $C(n,k)$ combinations, each copied in $O(k)$
- Space: $O(k)$ recursion depth

In [4]:
def combinations(n, k):
    res = []

    def backtrack(start, path):
        if len(path) == k:
            res.append(path[:])
            return
        for i in range(start, n + 1):
            path.append(i)
            backtrack(i + 1, path)
            path.pop()

    backtrack(1, [])
    return res


print(combinations(4, 2))

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


#### Steps

`combinations(4, 2)` — choose 2 from {1, 2, 3, 4}, yielding $C(4,2) = 6$ combinations.

**State space tree:**

```mermaid
graph TD
    A["path=[]"] -->|"append 1, start=2"| B["path=[1]"]
    A -->|"append 2, start=3"| C["path=[2]"]
    A -->|"append 3, start=4"| D["path=[3]"]
    A -->|"append 4, start=5"| E["path=[4]"]
    B -->|"append 2"| F["[1,2] *"]
    B -->|"append 3"| G["[1,3] *"]
    B -->|"append 4"| H["[1,4] *"]
    C -->|"append 3"| I["[2,3] *"]
    C -->|"append 4"| J["[2,4] *"]
    D -->|"append 4"| K["[3,4] *"]
    E -->|"range empty"| L["return"]
```

`*` = len == k, path recorded. `start` advances to `i + 1` after each append, so elements always appear in ascending order. [2,1] can never be generated because 1 is only available when `start <= 1`.

**Undo trace** (first branch):

| Step | Action | path |
|------|--------|------|
| 1 | append 1 | [1] |
| 2 | append 2, record [1,2] | [1, 2] |
| 3 | undo 2 (pop) | [1] |
| 4 | append 3, record [1,3] | [1, 3] |
| 5 | undo 3 (pop) | [1] |
| 6 | append 4, record [1,4] | [1, 4] |
| 7 | undo 4 (pop) | [1] |
| 8 | undo 1 (pop) | [] |

Result: `[[1,2], [1,3], [1,4], [2,3], [2,4], [3,4]]`

## 6. N-Queens

Place $n$ queens on an $n \times n$ board so no two queens share a row, column, or diagonal.

- **Choice:** at each row $r$, try placing a queen in each column $c = 0, \ldots, n-1$
- **Constraint:** column $c$ is not in `cols`, diagonal index $r - c$ is not in `diag`, anti-diagonal index $r + c$ is not in `antiDiag`
- **Goal:** all $n$ rows have a queen placed

Since the algorithm places exactly one queen per row, row conflicts are impossible by construction. Three sets track the remaining conflict types:

- `cols`: occupied column indices
- `diag`: occupied diagonal indices, where two cells $(r_1, c_1)$ and $(r_2, c_2)$ lie on the same diagonal if and only if $r_1 - c_1 = r_2 - c_2$
- `antiDiag`: occupied anti-diagonal indices, where two cells share an anti-diagonal if and only if $r_1 + c_1 = r_2 + c_2$

```text
4x4 board diagonal indexing:

     c=0  c=1  c=2  c=3
r=0   0   -1   -2   -3      <- diag = r - c
r=1   1    0   -1   -2
r=2   2    1    0   -1
r=3   3    2    1    0

     c=0  c=1  c=2  c=3
r=0   0    1    2    3      <- antiDiag = r + c
r=1   1    2    3    4
r=2   2    3    4    5
r=3   3    4    5    6
```

- Time: $O(n!)$ -- at row 0 there are $n$ choices, row 1 at most $n-1$, etc.
- Space: $O(n^2)$ for the board + $O(n)$ for conflict sets and recursion stack

In [5]:
def solveNQueens(n):
    res = []
    board = [["." for _ in range(n)] for _ in range(n)]
    cols, diag, antiDiag = set(), set(), set()

    def backtrack(r):
        if r == n:
            res.append(["".join(row) for row in board])
            return
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in antiDiag:
                continue
            cols.add(c)
            diag.add(r - c)
            antiDiag.add(r + c)
            board[r][c] = "Q"
            backtrack(r + 1)
            board[r][c] = "."
            cols.discard(c)
            diag.discard(r - c)
            antiDiag.discard(r + c)

    backtrack(0)
    return res


for sol in solveNQueens(4):
    for row in sol:
        print(row)
    print()

.Q..
...Q
Q...
..Q.

..Q.
Q...
...Q
.Q..



#### Steps

`solveNQueens(4)` -- place 4 queens on a 4x4 board.

**Placement trace for Solution 1:**

| Step | Row | Col | cols | diag (r-c) | antiDiag (r+c) | Board row |
|------|-----|-----|------|------------|----------------|-----------|
| 1 | 0 | 1 | {1} | {-1} | {1} | `.Q..` |
| 2 | 1 | 3 | {1,3} | {-1,-2} | {1,4} | `...Q` |
| 3 | 2 | 0 | {0,1,3} | {-2,-1,2} | {1,2,4} | `Q...` |
| 4 | 3 | 2 | {0,1,2,3} | {-2,-1,1,2} | {1,2,4,5} | `..Q.` |

After recording Solution 1, all choices undo and the search continues to find Solution 2.

```text
Solution 1:     Solution 2:
  .Q..            ..Q.
  ...Q            Q...
  Q...            ...Q
  ..Q.            .Q..
```

Row 0 tries c=0 first but reaches a dead end at row 2 (all columns conflict). Backtracking undoes rows 1 and 0, then tries c=1 at row 0, which leads to Solution 1. After exhausting all continuations from c=1 at row 0, the search moves to c=2, eventually finding Solution 2.

## 7. Sudoku Solver

Fill a 9x9 grid so each row, column, and 3x3 box contains digits 1-9 exactly once.

- **Choice:** at each empty cell, try digits 1 through 9
- **Constraint:** digit $d$ must not already appear in the current row, column, or box. Track with three arrays of sets: `rows[r]`, `cols[c]`, `boxes[bi]` where `bi = (r // 3) * 3 + c // 3`
- **Goal:** all empty cells filled (return `True` to propagate success up the call stack)

Unlike subsets or permutations, where the goal is to collect every valid result, the Sudoku solver only needs one filled board where every row, column, and box constraint is satisfied. Once `backtrack` finds that board, it returns `True` at each level of the recursion, skipping all remaining branches.

- Time: $O(9^m)$ worst case, where $m$ = number of empty cells. In practice, constraint checks eliminate most candidates at each cell, so the actual number of nodes explored is far smaller.
- Space: $O(m)$ recursion depth + $O(81)$ for constraint sets

In [6]:
def solveSudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empty = []
    for r in range(9):
        for c in range(9):
            if board[r][c] != ".":
                d = int(board[r][c])
                rows[r].add(d)
                cols[c].add(d)
                boxes[(r // 3) * 3 + c // 3].add(d)
            else:
                empty.append((r, c))

    def backtrack(idx):
        if idx == len(empty):
            return True
        r, c = empty[idx]
        bi = (r // 3) * 3 + c // 3
        for d in range(1, 10):
            if d in rows[r] or d in cols[c] or d in boxes[bi]:
                continue
            board[r][c] = str(d)
            rows[r].add(d)
            cols[c].add(d)
            boxes[bi].add(d)
            if backtrack(idx + 1):
                return True
            board[r][c] = "."
            rows[r].discard(d)
            cols[c].discard(d)
            boxes[bi].discard(d)
        return False

    backtrack(0)
    return board


board = [
    ["5", "3", ".", ".", "7", ".", ".", ".", "."],
    ["6", ".", ".", "1", "9", "5", ".", ".", "."],
    [".", "9", "8", ".", ".", ".", ".", "6", "."],
    ["8", ".", ".", ".", "6", ".", ".", ".", "3"],
    ["4", ".", ".", "8", ".", "3", ".", ".", "1"],
    ["7", ".", ".", ".", "2", ".", ".", ".", "6"],
    [".", "6", ".", ".", ".", ".", "2", "8", "."],
    [".", ".", ".", "4", "1", "9", ".", ".", "5"],
    [".", ".", ".", ".", "8", ".", ".", "7", "9"],
]
solveSudoku(board)
for row in board:
    print(" ".join(row))

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


#### Steps

Partial trace of the first few empty cells in the Sudoku above. The first empty cell is (0,2).

| Step | Cell | Try d | Constraint check | Action |
|------|------|-------|-----------------|--------|
| 1 | (0,2) | 1 | 1 not in rows[0]={5,3,7}, cols[2]={8}, boxes[0]={5,3,6,9,8} | place 1 |
| 2 | (0,3) | 2 | 2 in boxes[1]={7,1,9,5,8,2} | skip |
| 2 | (0,3) | 6 | 6 not in rows[0], cols[3], boxes[1] | place 6 |
| 3 | (0,5) | ... | ... | continue |
| ... | ... | ... | dead end reached | backtrack: undo (0,3)=6, try next d |

At each empty cell, digits 1-9 are tried in order. If a digit violates a row, column, or box constraint, it is skipped. If no digit fits, the function returns `False`, causing the caller to undo its last placement and advance to the next candidate.

## 8. Patterns

| Pattern | Decision at each step | Branching | Time |
|---------|----------------------|-----------|------|
| **Subsets** | Include or exclude current element | 2 | $O(n \cdot 2^n)$ |
| **Permutations** | Pick one unused element for current position | $n, n-1, \ldots, 1$ | $O(n \cdot n!)$ |
| **Combinations** | Pick next element with start-index constraint | varies | $O(C(n,k) \cdot k)$ |
| **Constraint satisfaction** | Assign a value to the current slot, skip values that violate rules | varies | $O(b^d)$ with pruning |
| **Grid / path search** | Move to each unvisited neighbor | up to 4 | $O(4^{r \cdot c})$ with pruning |

### Generate Parentheses

Generate all valid combinations of $n$ pairs of parentheses.

- **Choice:** at each position, add `(` or `)`
- **Constraints:** open count must not exceed $n$; close count must not exceed current open count
- **Goal:** path length reaches $2n$
- Time: $O(\frac{4^n}{\sqrt{n}})$ (Catalan number growth)

In [7]:
def generateParenthesis(n):
    res = []

    def backtrack(path, openCnt, closeCnt):
        if len(path) == 2 * n:
            res.append("".join(path))
            return
        if openCnt < n:
            path.append("(")
            backtrack(path, openCnt + 1, closeCnt)
            path.pop()
        if closeCnt < openCnt:
            path.append(")")
            backtrack(path, openCnt, closeCnt + 1)
            path.pop()

    backtrack([], 0, 0)
    return res


print(generateParenthesis(3))

['((()))', '(()())', '(())()', '()(())', '()()()']


#### Steps

`generateParenthesis(2)` — generate all valid sequences of 2 pairs.

**State space tree:**

```mermaid
graph TD
    A["path='', o=0, c=0"] -->|"append ("| B["path='(', o=1, c=0"]
    B -->|"append ("| C["path='((', o=2, c=0"]
    B -->|"append )"| D["path='()', o=1, c=1"]
    C -->|"append )"| E["path='(()', o=2, c=1"]
    E -->|"append )"| F["'(())' * o=2, c=2"]
    D -->|"append ("| G["path='()(', o=2, c=1"]
    G -->|"append )"| H["'()()' * o=2, c=2"]
```

`*` = len == 2n, path recorded. Nodes where `open >= n` cannot append `(`; nodes where `close >= open` cannot append `)`.

**Undo trace:**

| Step | Action | path | open | close |
|------|--------|------|------|-------|
| 1 | append ( | ( | 1 | 0 |
| 2 | append ( | (( | 2 | 0 |
| 3 | append ) | (() | 2 | 1 |
| 4 | append ), record (()) | (()) | 2 | 2 |
| 5 | undo ) (pop) | (() | 2 | 1 |
| 6 | undo ) (pop) | (( | 2 | 0 |
| 7 | undo ( (pop) | ( | 1 | 0 |
| 8 | append ) | () | 1 | 1 |
| 9 | append ( | ()( | 2 | 1 |
| 10 | append ), record ()() | ()() | 2 | 2 |

Result for n=2: `["(())", "()()"]`

### Combination Sum

Given a list of candidate numbers and a target integer, find all unique combinations of candidates that sum to the target. Each candidate may be used an unlimited number of times.

- **Choice:** at each level, pick a candidate starting from index `start`. To allow reuse, recurse with the same `start` (not `start + 1`).
- **Constraint:** if the running total exceeds the target, abandon the current path
- **Goal:** running total equals the target

In [8]:
def combinationSum(candidates, target):
    res = []

    def backtrack(start, path, total):
        if total == target:
            res.append(path[:])
            return
        if total > target:
            return
        for i in range(start, len(candidates)):
            path.append(candidates[i])
            backtrack(i, path, total + candidates[i])
            path.pop()

    backtrack(0, [], 0)
    return res


print(combinationSum([2, 3, 6, 7], 7))

[[2, 2, 3], [7]]


#### Steps

`combinationSum([2, 3, 6, 7], 7)`:

| Step | start | path | total | Action |
|------|-------|------|-------|--------|
| 1 | 0 | [2] | 2 | pick 2, recurse with start=0 (same element reusable) |
| 2 | 0 | [2,2] | 4 | pick 2 again |
| 3 | 0 | [2,2,2] | 6 | pick 2 again |
| 4 | 0 | [2,2,2,2] | 8 | total > 7, abandon path |
| 5 | 1 | [2,2,3] | 7 | total == 7, record [2,2,3] |
| 6 | -- | ... | -- | backtrack through remaining branches |
| 7 | 3 | [7] | 7 | total == 7, record [7] |

Result: `[[2, 2, 3], [7]]`

Recursing with `start=i` instead of `start=i+1` allows the same candidate to appear multiple times in a combination. The `start` parameter still enforces non-decreasing order, so reorderings like [3,2,2] are never generated.

## 9. Reference

### Complexity Summary

| Problem | Time | Space | Notes |
|---------|------|-------|-------|
| Subsets | $O(n \cdot 2^n)$ | $O(n)$ | $2^n$ subsets, copy each in $O(n)$ |
| Permutations | $O(n \cdot n!)$ | $O(n)$ | $n!$ permutations, copy each in $O(n)$ |
| Combinations | $O(C(n,k) \cdot k)$ | $O(k)$ | $C(n,k)$ combinations, copy each in $O(k)$ |
| N-Queens | $O(n!)$ | $O(n^2)$ | pruning reduces effective branching from $n^n$ |
| Sudoku | $O(9^m)$ | $O(m)$ | $m$ = empty cells; constraint checks reduce branching per cell |
| Generate parentheses | $O(\frac{4^n}{\sqrt{n}})$ | $O(n)$ | Catalan number growth |
| Combination sum | $O(n^{t/m})$ | $O(t/m)$ | $t$ = target, $m$ = min candidate value |

### Backtracking vs Other Approaches

| Approach | When to prefer |
|----------|----------------|
| Backtracking | State space is large, mostly unique; constraints eliminate many branches |
| Top-down DP | Subproblems overlap; states can be hashed and reused |
| BFS | Shortest path or minimum number of steps needed |
| Greedy | Optimal substructure + greedy choice property hold |

### Pattern Quick Reference

| Pattern | Template variation | Time | Use-case | Problem keywords |
|---------|-------------------|------|----------|-----------------|
| Subsets / powerset | Include-exclude binary tree | $O(n \cdot 2^n)$ | Generate all subsets of a set | "subsets", "powerset", "all subsequences" |
| Permutations | Used-array, pick for each position | $O(n \cdot n!)$ | Generate all orderings | "permutations", "next permutation", "arrange" |
| Combinations | Start-index to enforce ascending order | $O(C(n,k) \cdot k)$ | Choose $k$ items from $n$ | "combinations", "choose", "$C(n,k)$" |
| Combination sum | Start-index + running total, abandon when total exceeds target | $O(n^{t/m})$ | Find sets summing to a target | "combination sum", "target sum", "coin combinations" |
| Generate parentheses | Open/close counters as constraints | $O(4^n / \sqrt{n})$ | Generate valid bracket sequences | "generate parentheses", "valid brackets" |
| N-Queens / constraint satisfaction | Row-by-row placement + conflict sets | $O(n!)$ | Place items under mutual exclusion rules | "n-queens", "sudoku", "graph coloring" |
| Grid path / word search | 4-directional DFS + visited marker | $O(4^{r \cdot c})$ | Search for paths or words in a grid | "word search", "path exists", "rat in maze" |

### Cross-References

- Word search on board (trie + backtracking): see `tries.ipynb` section 6
- DFS traversal fundamentals: see `trees.ipynb` section 3
- Recursion and call stack: see `stacks_queues.ipynb`